# Kavach on Evon v3.3 (Q4_K_M, Kaggle 2× T4): prompts v2 on 14 test calls

Loads `anilkumarlucky/gnani-evon-v3.3-30b-a3b-GGUF` (Q4_K_M) into `llama-server`, split across both T4s, and answers three questions for Kavach:

1. **Does it fit?** Full GPU offload first; if a T4 runs out of memory, the first N layers' MoE experts move to CPU RAM (`--n-cpu-moe`) and we retry.
2. **How fast?** Prompt (prefill) and generation tokens/s, one request and two at once (Prosecutor + Defender run in parallel).
3. **Do the Kavach roles work?** Prompts v2 on 7 synthetic scam calls and 7 genuine calls, thinking off: verdicts, Telugu warnings, time to verdict.

**Before Run all**
1. Settings → Accelerator: **GPU T4 ×2**. Settings → Internet: **on**.
2. Add-ons → Secrets → add `HF` = your Hugging Face token, and tick it for this notebook.

Everything is written to `/kaggle/working/evon_t4_report_v2.md` (+ `.json`) at the end; download that and bring it back.

In [ ]:
# 1. Settings
import os, json, time, subprocess, shutil, re, threading
from pathlib import Path

GGUF_REPO  = "anilkumarlucky/gnani-evon-v3.3-30b-a3b-GGUF"
GGUF_FILE  = "gnani-evon-v3.3-30b-a3b-Q4_K_M.gguf"
BASE_REPO  = "gnani/gnani-evon-v3.3-30b-a3b"      # chat_template.jinja lives here, not in tokenizer_config
LLAMA_SHA  = "b3fa699"                             # the llama.cpp commit the GGUF was converted with
SRC        = Path("/tmp/llama.cpp")
BIN        = Path("/kaggle/working/llama-bin")     # keep binaries so a rerun in the same session skips the build
MODEL_DIR  = Path("/tmp/evon")                     # /kaggle/working is capped at 20 GB, so the model goes to /tmp
PORT       = 8080
URL        = f"http://127.0.0.1:{PORT}"
CTX, SLOTS = 16384, 2                              # 8k tokens per slot, two slots for parallel roles
CPU_MOE_STEPS = [0, 6, 12, 20]                     # retry ladder if the T4s run out of memory

REPORT = {"started": time.strftime("%Y-%m-%d %H:%M:%S")}
def note(k, v):
    REPORT[k] = v
    print(f"{k}: {v}")

In [ ]:
# 2. Hardware check
!nvidia-smi --query-gpu=index,name,memory.total,memory.used --format=csv
!nvcc --version | tail -2
!free -g | head -2; df -h /tmp /kaggle/working | tail -2
gpus = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.split("\n")
gpus = [g for g in gpus if g.strip()]
note("gpus", gpus)
assert len(gpus) == 2, "Set Accelerator to GPU T4 x2"

## Build llama-server with CUDA (about 15–25 min, once per session)

Only the T4's architecture (sm_75) is compiled, which cuts the build time a lot. Static linking keeps the binary self-contained. Kaggle ships no CUDA stub library, so CMake is pointed at the real driver in /usr/local/nvidia/lib64.

In [ ]:
# 3. Build
SERVER = BIN / "llama-server"
if not SERVER.exists():
    t = time.time()
    if not SRC.exists():
        !git clone -q --filter=blob:none https://github.com/ggml-org/llama.cpp {SRC}
    !git -C {SRC} checkout -q {LLAMA_SHA}
    !cd {SRC} && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DBUILD_SHARED_LIBS=OFF -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release -DCUDA_cuda_driver_LIBRARY=/usr/local/nvidia/lib64/libcuda.so > /tmp/cmake.log 2>&1 || tail -30 /tmp/cmake.log
    !cd {SRC} && cmake --build build --target llama-server -j {os.cpu_count()} > /tmp/build.log 2>&1 || tail -40 /tmp/build.log
    BIN.mkdir(parents=True, exist_ok=True)
    shutil.copy(SRC / "build/bin/llama-server", SERVER)
    note("build_minutes", round((time.time() - t) / 60, 1))
assert SERVER.exists(), "build failed, see the log tail above"
note("llama_cpp", LLAMA_SHA)

In [ ]:
# 4. Download the Q4_K_M GGUF (private repo) and Evon's chat template
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF")
t = time.time()
MODEL = Path(hf_hub_download(GGUF_REPO, GGUF_FILE, local_dir=MODEL_DIR))
TEMPLATE = Path(hf_hub_download(BASE_REPO, "chat_template.jinja", local_dir=MODEL_DIR))
note("model_gb", round(MODEL.stat().st_size / 1e9, 2))
note("download_minutes", round((time.time() - t) / 60, 1))

## Start the server

`-sm layer -ts 1,1` puts half the layers on each T4. If loading fails with an out-of-memory error, the next step of `CPU_MOE_STEPS` keeps that many layers' experts in CPU RAM (slower, but only 6 of 128 experts are active per token).

In [ ]:
# 5. Start llama-server, stepping down the GPU load until it fits
import requests

def gpu_mem():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout.strip().splitlines()
    return [f"{int(u)/1024:.1f}/{int(t)/1024:.1f} GB" for u, t in (l.split(", ") for l in out)]

def start_server(n_cpu_moe):
    log = open(f"/tmp/server_cpumoe{n_cpu_moe}.log", "w")
    cmd = [str(SERVER), "-m", str(MODEL), "--chat-template-file", str(TEMPLATE),
           "-ngl", "999", "-sm", "layer", "-ts", "1,1", "-c", str(CTX), "-np", str(SLOTS),
           "-fa", "auto", "--host", "127.0.0.1", "--port", str(PORT), "--metrics"]
    if n_cpu_moe:
        cmd += ["--n-cpu-moe", str(n_cpu_moe)]
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    t = time.time()
    while time.time() - t < 900:
        if proc.poll() is not None:
            return None, log.name
        try:
            if requests.get(f"{URL}/health", timeout=2).status_code == 200:
                return proc, log.name
        except requests.RequestException:
            pass
        time.sleep(5)
    proc.kill()
    return None, log.name

server = None
for n in CPU_MOE_STEPS:
    t = time.time()
    server, logname = start_server(n)
    if server:
        note("n_cpu_moe", n)
        note("load_seconds", round(time.time() - t))
        note("vram_after_load", gpu_mem())
        break
    tail = open(logname).read()[-1500:]
    print(f"--- n_cpu_moe={n} failed; log tail:\n{tail}")
    if not re.search(r"out of memory|failed to allocate|cudaMalloc", tail, re.I):
        raise SystemExit("Server failed for a reason other than memory; see the log tail above.")
assert server, "Did not fit even with the largest --n-cpu-moe step"

In [ ]:
# 6. Client helper. Thinking is on by default in Evon's template, so every call says which it wants.
def chat(messages, max_tokens=512, thinking=False, schema=None, temperature=0.3, cache_prompt=True):
    body = {"messages": messages, "max_tokens": max_tokens, "temperature": temperature,
            "chat_template_kwargs": {"enable_thinking": thinking}, "cache_prompt": cache_prompt}
    if schema:
        body["response_format"] = {"type": "json_object", "schema": schema}
    t = time.time()
    r = requests.post(f"{URL}/v1/chat/completions", json=body, timeout=900)
    r.raise_for_status()
    j = r.json()
    msg = j["choices"][0]["message"]
    tm = j.get("timings", {})
    return {"content": msg.get("content") or "", "reasoning": msg.get("reasoning_content") or "",
            "wall_s": round(time.time() - t, 2),
            "prompt_tokens": tm.get("prompt_n"), "prompt_tps": round(tm.get("prompt_per_second", 0), 1),
            "gen_tokens": tm.get("predicted_n"), "gen_tps": round(tm.get("predicted_per_second", 0), 1),
            "finish": j["choices"][0].get("finish_reason")}

def n_tokens(text):
    return len(requests.post(f"{URL}/tokenize", json={"content": text}).json()["tokens"])

warm = chat([{"role": "user", "content": "Say OK."}], max_tokens=8)
print(warm)

## Speed

In [ ]:
# 7. Generation and prefill speed, then two requests at once
from concurrent.futures import ThreadPoolExecutor

speed = {}
r = chat([{"role": "user", "content": "Write 200 words about why people should never share an OTP on a phone call."}],
         max_tokens=300, temperature=0.7, cache_prompt=False)
speed["single_gen"] = {k: r[k] for k in ("prompt_tokens", "prompt_tps", "gen_tokens", "gen_tps", "wall_s")}

long_text = ("A bank customer receives a call. The caller claims to be from the cyber police and says a parcel "
             "in the customer's name contained drugs. ") * 120
r = chat([{"role": "user", "content": long_text + "\nSummarise the above in one sentence."}],
         max_tokens=40, cache_prompt=False)
speed["prefill_long"] = {k: r[k] for k in ("prompt_tokens", "prompt_tps", "gen_tokens", "gen_tps", "wall_s")}

prompts = ["List five signs of a phone scam.", "List five signs that a bank call is genuine."]
t = time.time()
with ThreadPoolExecutor(2) as ex:
    rs = list(ex.map(lambda p: chat([{"role": "user", "content": p}], max_tokens=200, temperature=0.7,
                                    cache_prompt=False), prompts))
speed["two_parallel"] = {"wall_s": round(time.time() - t, 2), "per_request_gen_tps": [x["gen_tps"] for x in rs],
                         "total_gen_tokens": sum(x["gen_tokens"] for x in rs)}
speed["two_parallel"]["aggregate_gen_tps"] = round(speed["two_parallel"]["total_gen_tokens"] / speed["two_parallel"]["wall_s"], 1)

te = "మీ ఆధార్ నంబర్‌తో ఒక పార్సెల్‌లో డ్రగ్స్ దొరికాయి. ఈ విషయం ఎవరికీ చెప్పకూడదు, వెంటనే వీడియో కాల్‌లో ఉండండి."
speed["telugu_tokens_per_word"] = round(n_tokens(te) / len(te.split()), 2)
note("speed", speed)
note("vram_after_speed", gpu_mem())

## Kavach roles, v2 (thinking off throughout)

Changes after the first run, where a genuine bank call was flagged and the warning repeated the caller's words:

- **Neutral framing.** The system prompt says most calls are genuine; nothing tells the model to find a scam.
- **Structured evidence.** Prosecutor and Defender return grammar-constrained JSON with a quote for every claim. Quotes that are not actually in the transcript are dropped in code before the Judge sees them.
- **Explicit rules for the Judge**, including what does *not* count: a caller saying they will never ask for an OTP, or pointing to the official number.
- **The warning never sees the transcript.** It gets the verdict and tactic labels only, plus one example, so it cannot echo the caller. SAFE calls get no warning at all: Kavach stays silent.
- **14 short synthetic calls**: 7 scams of different kinds and 7 genuine calls, several of them deliberately close to a scam (fraud alert, pension deadline, cash-on-delivery).

In [ ]:
# 8. Test calls: hand-written, Telugu-English mix as Prisma is expected to return it
CALLS = [
 ("scam", "digital_arrest", '''Caller: హలో, నేను Mumbai Cyber Crime police station నుంచి Inspector Sharma మాట్లాడుతున్నాను.
Caller: మీ ఆధార్ నంబర్‌తో ఒక parcel లో drugs దొరికాయి. మీ మీద FIR register అయింది.
User: అయ్యో, నాకు ఏమీ తెలియదు సార్.
Caller: మీరు ఇప్పుడే video call లో ఉండాలి, ఈ విషయం ఎవరికీ చెప్పకూడదు, లేకపోతే arrest warrant issue చేస్తాం.
Caller: మీ savings మొత్తం RBI safe account కి transfer చేయండి, investigation తర్వాత return చేస్తాం.'''),
 ("scam", "kyc_otp", '''Caller: నమస్తే అమ్మా, నేను SBI head office నుంచి మాట్లాడుతున్నాను.
Caller: మీ KYC expire అయింది, ఈరోజు update చేయకపోతే మీ account block అవుతుంది.
User: అవునా, ఏం చేయాలి?
Caller: ఇప్పుడు మీ phone కి ఒక OTP వస్తుంది, అది నాకు చెప్పండి, నేను ఇక్కడే update చేస్తాను.'''),
 ("scam", "power_cut_upi", '''Caller: TSSPDCL electricity office నుంచి call చేస్తున్నాం.
Caller: మీ last month bill pending ఉంది, ఈ రాత్రి 9:30 కి మీ power connection cut చేస్తాం.
User: నేను bill కట్టేశాను కదా.
Caller: Update కాలేదు సార్. వెంటనే ఈ number కి UPI లో 1,250 rupees pay చేయండి, లేకపోతే connection cut అవుతుంది.'''),
 ("scam", "grandson_emergency", '''Caller: నానమ్మా, నేను రవి. నాకు accident అయింది, hospital లో ఉన్నాను.
User: అయ్యో రవి, ఏమైంది? నీ గొంతు వేరేగా ఉంది.
Caller: జలుబు నానమ్మా. అమ్మకి చెప్పకు, టెన్షన్ పడుతుంది. Doctor కి వెంటనే 20,000 కట్టాలి.
Caller: నేను పంపే number కి Google Pay లో పంపు, ప్లీజ్ ఎవరికీ చెప్పకు.'''),
 ("scam", "remote_access", '''Caller: Sir, మీ bank app లో security problem వచ్చింది, మీ money risk లో ఉంది.
Caller: Play Store లో AnyDesk అనే app download చేయండి.
User: ఎందుకు?
Caller: మా technical team మీ phone check చేస్తుంది. App open చేసి అందులో వచ్చే 9 digit code నాకు చెప్పండి.'''),
 ("scam", "lottery_fee", '''Caller: Congratulations సార్! KBC lucky draw లో మీ number కి 25 లక్షలు వచ్చాయి.
User: నేను ఏ lottery కొనలేదే.
Caller: ఇది Jio customers కి free draw సార్. Prize release కావాలంటే ముందు 5,000 rupees processing fee కట్టాలి.
Caller: ఈరోజే కట్టకపోతే prize next person కి వెళ్తుంది.'''),
 ("scam", "customs_parcel", '''Caller: నేను FedEx courier నుంచి మాట్లాడుతున్నాను. మీ పేరు మీద Taiwan కి వెళ్లే parcel customs లో ఆపారు.
Caller: అందులో fake passports ఉన్నాయి. నేను మిమ్మల్ని customs officer కి connect చేస్తాను.
User: నేను ఏ parcel పంపలేదు.
Caller: అయితే case close చేయాలంటే 48,000 fine ఇప్పుడే online లో కట్టాలి, మీ card details చెప్పండి.'''),
 ("genuine", "card_delivery", '''Caller: నమస్కారం, నేను SBI Card customer care నుంచి మాట్లాడుతున్నాను.
Caller: మీ కొత్త credit card రేపు courier లో వస్తుంది. Delivery సమయంలో ఏదైనా ID proof చూపించండి.
User: సరే, ఏమైనా OTP చెప్పాలా?
Caller: లేదు సార్, మేము ఎప్పుడూ OTP కానీ PIN కానీ అడగము. ఏదైనా సందేహం ఉంటే card వెనుక ఉన్న number కి call చేయండి.
User: సరే, thank you.'''),
 ("genuine", "fraud_alert", '''Caller: నేను HDFC Bank fraud monitoring team నుంచి మాట్లాడుతున్నాను.
Caller: మీ card మీద ఇప్పుడే 8,500 rupees online transaction వచ్చింది. అది మీరు చేశారా?
User: లేదు, నేను చేయలేదు.
Caller: సరే, మీ card ని మేము temporary గా block చేశాం. కొత్త card కోసం మీ దగ్గరలోని branch కి వెళ్ళండి లేదా app లో request పెట్టండి. మేము మిమ్మల్ని OTP లేదా PIN అడగము.'''),
 ("genuine", "hospital_reminder", '''Caller: నమస్కారం, Apollo hospital నుంచి call చేస్తున్నాం.
Caller: రేపు ఉదయం 10 గంటలకు Dr. Reddy గారితో మీ appointment ఉంది.
Caller: Blood test ఉంది కాబట్టి ఖాళీ కడుపుతో రండి, పాత reports కూడా తీసుకురండి.
User: సరే, వస్తాను.'''),
 ("genuine", "grandson_visit", '''Caller: నానమ్మా, నేను రవి. ఎలా ఉన్నావు?
User: బాగున్నాను రా, నువ్వు?
Caller: బాగున్నా. ఆదివారం అమ్మతో కలిసి ఇంటికి వస్తున్నాం. నీకు ఏమైనా తీసుకురావాలా?
User: ఏమీ వద్దు, మీరు రండి చాలు.'''),
 ("genuine", "power_maintenance", '''Caller: TSSPDCL నుంచి సమాచారం.
Caller: రేపు line maintenance వల్ల ఉదయం 10 నుంచి మధ్యాహ్నం 2 వరకు మీ area లో power ఉండదు.
Caller: మీరు ఎలాంటి payment చేయాల్సిన అవసరం లేదు. ఇది సమాచారం మాత్రమే.'''),
 ("genuine", "pension_life_certificate", '''Caller: నమస్కారం సార్, నేను Canara Bank, Ameerpet branch నుంచి మాట్లాడుతున్నాను.
Caller: మీ pension కోసం life certificate November 30 లోపు submit చేయాలి.
User: ఎలా చేయాలి?
Caller: Aadhaar card తీసుకుని branch కి రండి, లేదా Jeevan Pramaan app లో మీరే చేయవచ్చు. Phone లో ఏ details చెప్పకండి.'''),
 ("genuine", "cash_on_delivery", '''Caller: Hello అమ్మా, Amazon delivery. మీ ఇంటి gate దగ్గర ఉన్నాను.
Caller: మీ order cash on delivery, 450 rupees అవుతుంది.
User: సరే, కిందకి వస్తున్నా.
Caller: సరే అమ్మా, wait చేస్తాను.'''),
]
print(len(CALLS), "calls:", sum(k == "scam" for k, _, _ in CALLS), "scam,", sum(k == "genuine" for k, _, _ in CALLS), "genuine")

In [ ]:
# 9. Role prompts (v2)
SYS = ("You analyse phone-call transcripts for Kavach, which helps elderly Telugu speakers. "
       "Most calls people receive are genuine. Judge only from what the transcript actually says, and quote lines exactly.")

TACTICS = ["money_or_fee_request", "otp_pin_password_request", "card_or_bank_details_request",
           "remote_access_app", "arrest_or_legal_threat", "account_block_or_disconnection_threat",
           "secrecy_demand", "impersonation_with_pressure", "suspicious_link"]

PRO_SCHEMA = {"type": "object", "properties": {"evidence": {"type": "array", "items": {"type": "object", "properties": {
    "tactic": {"type": "string", "enum": TACTICS}, "quote": {"type": "string"}}, "required": ["tactic", "quote"]}}},
    "required": ["evidence"]}
DEF_SCHEMA = {"type": "object", "properties": {
    "benign_signals": {"type": "array", "items": {"type": "object", "properties": {
        "signal": {"type": "string"}, "quote": {"type": "string"}}, "required": ["signal", "quote"]}},
    "unexplained_quotes": {"type": "array", "items": {"type": "string"}}},
    "required": ["benign_signals", "unexplained_quotes"]}
JUDGE_SCHEMA = {"type": "object", "properties": {
    "verdict": {"type": "string", "enum": ["SCAM", "SUSPICIOUS", "SAFE"]},
    "confidence": {"type": "number"},
    "tactics": {"type": "array", "items": {"type": "string", "enum": TACTICS}},
    "reason": {"type": "string"}},
    "required": ["verdict", "confidence", "tactics", "reason"]}

def prosecutor(call):
    return [{"role": "system", "content": SYS}, {"role": "user", "content":
        f"Transcript:\n{call}\n\nList every line where the CALLER actually does one of these: {', '.join(TACTICS)}.\n"
        "Rules: a tactic counts only if the caller does it, not if they deny it. "
        "'We never ask for OTP/PIN' is NOT a tactic. Pointing the listener to the official number, the bank branch or the bank's own app is NOT a tactic. "
        "Asking for payment in person on delivery of something the listener ordered is NOT a tactic. "
        "Copy each quote exactly from the transcript. If there is nothing, return an empty list."}]

def defender(call):
    return [{"role": "system", "content": SYS}, {"role": "user", "content":
        f"Transcript:\n{call}\n\nList the signs that this call is genuine, each with an exact quote: for example the caller only gives information, "
        "says they will not ask for OTP/PIN, sends the listener to an official number, branch or app, or the call concerns something the listener expects. "
        "Then list, as exact quotes, any caller lines that no genuine caller would say. Use empty lists where there is nothing."}]

def judge(call, pro, dfn):
    return [{"role": "system", "content": SYS}, {"role": "user", "content":
        f"Transcript:\n{call}\n\nVerified scam evidence: {json.dumps(pro, ensure_ascii=False)}\n"
        f"Verified genuine signals: {json.dumps(dfn, ensure_ascii=False)}\n\n"
        "Decide:\n"
        "- SCAM: the caller asks for money or a fee, an OTP/PIN/password, card or bank details, or a remote-access app; "
        "or threatens arrest, legal action, account block or disconnection to push the listener to act now.\n"
        "- SUSPICIOUS: pressure, secrecy or impersonation without any such request yet, or a link to open.\n"
        "- SAFE: none of the above. A caller who says they are from a bank, hospital or utility is SAFE if they ask for nothing "
        "and point to official channels. Cash on delivery for an order is SAFE.\n"
        "Return JSON: verdict, confidence 0-1, tactics (labels from the evidence that you accept; empty for SAFE), reason (one English sentence)."}]

TACTIC_TE = {"money_or_fee_request": "డబ్బులు అడుగుతున్నారు", "otp_pin_password_request": "OTP లేదా PIN అడుగుతున్నారు",
             "card_or_bank_details_request": "card లేదా bank వివరాలు అడుగుతున్నారు", "remote_access_app": "phone లో app వేయమంటున్నారు",
             "arrest_or_legal_threat": "arrest చేస్తామని భయపెడుతున్నారు", "account_block_or_disconnection_threat": "account లేదా connection ఆపేస్తామని భయపెడుతున్నారు",
             "secrecy_demand": "ఎవరికీ చెప్పొద్దంటున్నారు", "impersonation_with_pressure": "అధికారులమని చెప్పి తొందర పెడుతున్నారు",
             "suspicious_link": "link పంపుతున్నారు"}

def warning(verdict, tactics):
    hints = "; ".join(TACTIC_TE[t] for t in tactics if t in TACTIC_TE) or "అనుమానంగా ఉంది"
    return [{"role": "system", "content": "You write one short spoken warning in simple Telugu (Telugu script) for an elderly listener. "
             "English words common in Telugu speech (OTP, bank, app, phone) may stay in English. Never repeat what the caller said."},
            {"role": "user", "content": "Verdict: SCAM. Signs: OTP లేదా PIN అడుగుతున్నారు"},
            {"role": "assistant", "content": "అమ్మా, ఇది మోసం కాల్. OTP ఎవరికీ చెప్పకండి, వెంటనే ఫోన్ పెట్టేయండి."},
            {"role": "user", "content": f"Verdict: {verdict}. Signs: {hints}"}]

def reporter(call, verdict, tactics):
    return [{"role": "system", "content": SYS}, {"role": "user", "content":
        f"Transcript:\n{call}\n\nVerdict: {verdict}; tactics: {', '.join(tactics)}.\n"
        "Draft a short complaint the family can file on the 1930 helpline / cybercrime.gov.in: what the caller claimed, what they demanded, and the tactics. "
        "English, at most 80 words, then the same summary in Telugu."}]

In [ ]:
# 10. Helpers: drop quotes the model invented, measure Telugu and echoing
def _norm(s):
    return re.sub(r"[^\wఀ-౿]+", " ", s.lower()).strip()

def in_transcript(quote, call):
    # a quote shortened with "..." passes if every piece is in the transcript
    parts = [_norm(p) for p in re.split(r"\.\.\.|…", quote)]
    parts = [p for p in parts if p]
    return bool(parts) and sum(map(len, parts)) >= 6 and all(p in _norm(call) for p in parts)

def telugu_ratio(s):
    letters = [c for c in s if c.isalpha()]
    return round(sum('ఀ' <= c <= '౿' for c in letters) / max(len(letters), 1), 2)

def echoes(text, call, n=3):
    # True if the warning repeats any 3-word run of the CALLER's lines
    caller = " ".join(l.split(":", 1)[1] for l in call.splitlines() if l.startswith("Caller:"))
    grams = lambda s: {tuple(w[i:i + n]) for w in [_norm(s).split()] for i in range(len(w) - n + 1)}
    return bool(grams(text) & grams(caller))

def parse(r):
    try:
        return json.loads(r["content"])
    except json.JSONDecodeError:
        return None

In [ ]:
# 11. Run all 14 calls
results = []
for kind, name, call in CALLS:
    t = time.time()
    with ThreadPoolExecutor(2) as ex:
        pro_r, dfn_r = ex.map(lambda a: chat(a[0], max_tokens=500, schema=a[1], temperature=0.1),
                              [(prosecutor(call), PRO_SCHEMA), (defender(call), DEF_SCHEMA)])
    pro, dfn = parse(pro_r) or {"evidence": []}, parse(dfn_r) or {"benign_signals": [], "unexplained_quotes": []}
    pro_ok = [e for e in pro["evidence"] if in_transcript(e["quote"], call)]
    dfn_ok = [e for e in dfn["benign_signals"] if in_transcript(e["quote"], call)]
    jd_r = chat(judge(call, pro_ok, dfn_ok), max_tokens=300, schema=JUDGE_SCHEMA, temperature=0.1)
    jd = parse(jd_r) or {"verdict": "PARSE_ERROR", "tactics": []}
    t_verdict = time.time() - t
    flagged = jd["verdict"] in ("SCAM", "SUSPICIOUS")
    wn = chat(warning(jd["verdict"], jd["tactics"]), max_tokens=120, temperature=0.3)["content"].strip() if flagged else ""
    rp = chat(reporter(call, jd["verdict"], jd["tactics"]), max_tokens=500)["content"].strip() if flagged else ""
    results.append({
        "kind": kind, "name": name, "verdict": jd.get("verdict"), "confidence": jd.get("confidence"),
        "correct": flagged == (kind == "scam"), "tactics": jd.get("tactics"), "reason": jd.get("reason"),
        "seconds_to_verdict": round(t_verdict, 1),
        "evidence_kept": len(pro_ok), "evidence_dropped": len(pro["evidence"]) - len(pro_ok),
        "benign_kept": len(dfn_ok), "benign_dropped": len(dfn["benign_signals"]) - len(dfn_ok),
        "prosecutor": pro, "defender": dfn,
        "warning_te": wn, "warning_telugu_ratio": telugu_ratio(wn) if wn else None,
        "warning_echoes_caller": echoes(wn, call) if wn else None, "report": rp})
    r = results[-1]
    print(f"{'OK ' if r['correct'] else 'BAD'} {kind:7s} {name:24s} {r['verdict']:10s} {r['seconds_to_verdict']:5.1f}s  {wn}")
note("vram_after_roles", gpu_mem())

In [ ]:
# 12. Summary
scam = [r for r in results if r["kind"] == "scam"]
gen = [r for r in results if r["kind"] == "genuine"]
warned = [r for r in results if r["warning_te"]]
summary = {
    "scams_flagged": f"{sum(r['correct'] for r in scam)}/{len(scam)}",
    "genuine_left_alone": f"{sum(r['correct'] for r in gen)}/{len(gen)}",
    "median_seconds_to_verdict": sorted(r["seconds_to_verdict"] for r in results)[len(results) // 2],
    "max_seconds_to_verdict": max(r["seconds_to_verdict"] for r in results),
    "warnings_mostly_telugu": f"{sum((r['warning_telugu_ratio'] or 0) >= 0.6 for r in warned)}/{len(warned)}",
    "warnings_echoing_caller": f"{sum(bool(r['warning_echoes_caller']) for r in warned)}/{len(warned)}",
    "invented_quotes_dropped": sum(r["evidence_dropped"] + r["benign_dropped"] for r in results)}
note("summary", summary)

In [ ]:
# 13. Write the report and stop the server
REPORT["results"] = results
REPORT["finished"] = time.strftime("%Y-%m-%d %H:%M:%S")
Path("/kaggle/working/evon_t4_report_v2.json").write_text(json.dumps(REPORT, ensure_ascii=False, indent=2), encoding="utf-8")

sp = REPORT["speed"]
lines = [f"# Kavach prompts v2 on Evon v3.3 Q4_K_M, Kaggle 2x T4 ({REPORT['finished']})", "",
         f"- llama.cpp {REPORT['llama_cpp']}, --n-cpu-moe {REPORT['n_cpu_moe']}, VRAM {REPORT['vram_after_roles']}",
         f"- Generation {sp['single_gen']['gen_tps']} tok/s; prefill {sp['prefill_long']['prompt_tps']} tok/s",
         "", "## Summary", "", "```", json.dumps(REPORT["summary"], indent=2, ensure_ascii=False), "```", "",
         "| | call | verdict | conf | s | tactics | warning (te %, echo) |", "|---|---|---|---|---|---|---|"]
for r in results:
    w = f"{r['warning_te']} ({r['warning_telugu_ratio']:.0%}, {'ECHO' if r['warning_echoes_caller'] else 'ok'})" if r["warning_te"] else "silent"
    lines.append(f"| {'OK' if r['correct'] else '**WRONG**'} | {r['kind']}/{r['name']} | {r['verdict']} | {r['confidence']} | "
                 f"{r['seconds_to_verdict']} | {', '.join(r['tactics'] or [])} | {w} |")
for r in results:
    lines += ["", f"### {r['kind']}/{r['name']}: {r['verdict']}", "", f"Reason: {r['reason']}", "",
              f"Evidence kept {r['evidence_kept']}, dropped {r['evidence_dropped']}; genuine signals kept {r['benign_kept']}, dropped {r['benign_dropped']}"]
    if r["report"]:
        lines += ["", "Report:", "", r["report"]]
Path("/kaggle/working/evon_t4_report_v2.md").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines[:30]))

server.terminate()
print("\nwrote /kaggle/working/evon_t4_report_v2.md and .json")